In [3]:
import collections
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

texts = [
    "отличный сервис спасибо за помощь", "приложение лагает и вылетает",
    "быстрый перевод денег без комиссии", "украли деньги со счета мошенники",
    "карта работает отлично кэшбэк пришел", "ужасная техподдержка не отвечают",
    "оформил ипотеку под низкий процент", "подозрительное списание без смс"
] * 100

labels = [0, 1, 0, 2, 0, 1, 0, 2] * 100  # 0: Хорошо, 1: Сервис/Баги, 2: Мошенники

class SimpleVocab:
    def __init__(self, texts):
        tokens = [w for text in texts for w in text.split()]
        counts = collections.Counter(tokens)
        self.w2i = {"<pad>": 0, "<unk>": 1}
        for w in counts:
            self.w2i[w] = len(self.w2i)
    def encode(self, text):
        return [self.w2i.get(w, 1) for w in text.split()]
    def __len__(self):
        return len(self.w2i)

vocab = SimpleVocab(texts)

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.data = [vocab.encode(t) for t in texts]
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels, offsets = [], [], [0]
    for _text, _label in batch:
        labels.append(_label)
        t = torch.tensor(_text, dtype=torch.long)
        texts.append(t)
        offsets.append(t.size(0))
    return torch.cat(texts), torch.tensor(offsets[:-1]).cumsum(dim=0), torch.tensor(labels)

loader = DataLoader(TextDataset(texts, labels, vocab), batch_size=16, shuffle=True, collate_fn=collate_fn)

In [6]:
from torch.optim.lr_scheduler import CosineAnnealingLR

class DeepTextMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden1, hidden2, num_classes, dropout=0.2):
        super().__init__()
        # входной слой
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode = "mean")

        # скрытый слой 1
        self.fc1 = nn.Linear(embed_dim, hidden1)
        self.act1 = nn.GELU()
        self.dropout1 = nn.Dropout(dropout)

        # скрытый слой 2
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.act2 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout)

        # выходной слой 
        self.fc3 = nn.Linear(hidden2, num_classes)

        # инилизация весов
        self._init_weights()
        
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, text, offsets):
        x = self.embedding(text, offsets)
        x = self.fc1(x)
        x = self.act1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.act2(x)
        x = self.dropout2(x)
        logits = self.fc3(x)
        return logits

# использовать видеокарту для обучения если она доступна, иначе процессор
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# инилизируем модель
model = DeepTextMLP(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden1 = 32,
    hidden2 = 16,
    num_classes = 3,
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-2)

# цикл обучения
epochs = 50
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    # переводим в режим обучения (активирует Dropout)
    model.train() 
    total_loss = 0.0

    for text, offsets, targets in loader:
        text, offsets, targets = text.to(device), offsets.to(device), targets.to(device)

        optimizer.zero_grad()

        outputs = model(text, offsets)

        loss = criterion(outputs, targets)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    current_lr = scheduler.get_last_lr()[0]
    avg_loss = total_loss / len(loader)
    if (epoch+1) % 5 == 0:
        print(f"epoch {epoch+1:02d}/{epochs} | Loss {avg_loss:.4f} | LR {current_lr:.6f}")       

epoch 05/50 | Loss 0.0004 | LR 0.009755
epoch 10/50 | Loss 0.0001 | LR 0.009045
epoch 15/50 | Loss 0.0002 | LR 0.007939
epoch 20/50 | Loss 0.0001 | LR 0.006545
epoch 25/50 | Loss 0.0001 | LR 0.005000
epoch 30/50 | Loss 0.0000 | LR 0.003455
epoch 35/50 | Loss 0.0039 | LR 0.002061
epoch 40/50 | Loss 0.0000 | LR 0.000955
epoch 45/50 | Loss 0.0000 | LR 0.000245
epoch 50/50 | Loss 0.0000 | LR 0.000000
